# Notebook 02: TransE Baseline Scoring

**Project:** Explainable Knowledge Graph-Based Drug Repurposing for Acute Brain Injury: Extending XAIPath to DRKG  
**Author:** Maha Attique  
**Date:** July 2026  

Score all candidate drugs against target acute brain injury disease nodes using DRKG's pretrained TransE embeddings. This is **Baseline 1** in the benchmarking pipeline.

TransE models relationships as translations in embedding space:
$$\mathbf{score} = \gamma - ||\mathbf{h} + \mathbf{r} - \mathbf{t}||_2$$

where $\mathbf{h}$ is the drug embedding, $\mathbf{r}$ is the treatment relation embedding, and $\mathbf{t}$ is the disease embedding. Higher scores mean stronger predicted treatment relationships.

## 1. Setup and Imports

In [1]:

import numpy as np
import pandas as pd
import csv
import torch
import torch.nn.functional as fn
import sys
sys.path.insert(1, '../utils')
from utils import download_and_extract

download_and_extract()


## 2. Target Disease Nodes and Treatment Relations

Target disease nodes verified against DRKG `embed/entities.tsv`. Treatment relation types follow DRKG's original COVID-19 repurposing methodology.

In [2]:
brain_injury_disease_list = [
    'Disease::MESH:D020521',  # Stroke
    'Disease::MESH:D002544',  # Cerebral Infarction (ischemic stroke)
    'Disease::MESH:D020300',  # Intracranial Hemorrhages
    'Disease::MESH:D020520',  # Intracranial Hemorrhage, Hypertensive
    'Disease::MESH:D002538',  # Cerebral Hemorrhage
    'Disease::MESH:D001930',  # Brain Injuries
    'Disease::MESH:D006470',  # Hemorrhage
]

treatment_relations = [
    'Hetionet::CtD::Compound:Disease',
    'GNBR::T::Compound:Disease'
]

# print(f"Target diseases: {len(brain_injury_disease_list)}")
# print(f"Treatment relations: {len(treatment_relations)}")

## Load candidate drugs
FDA-approved DrugBank compounds (mol weight >= 250), same list used in DRKG's original COVID-19 repurposing notebook.

In [3]:
drug_list = []
with open("../drug_repurpose/infer_drug.tsv", newline='', encoding='utf-8') as f:
    reader = csv.DictReader(f, delimiter='\t', fieldnames=['drug', 'ids'])
    for row in reader:
        drug_list.append(row['drug'])

len(drug_list)

8104

## Load entity/relation maps and pretrained embeddings

In [4]:
entity_map = {}
entity_id_map = {}
relation_map = {}

with open('../data/embed/entities.tsv', newline='', encoding='utf-8') as f:
    for row in csv.DictReader(f, delimiter='\t', fieldnames=['name', 'id']):
        entity_map[row['name']] = int(row['id'])
        entity_id_map[int(row['id'])] = row['name']

with open('../data/embed/relations.tsv', newline='', encoding='utf-8') as f:
    for row in csv.DictReader(f, delimiter='\t', fieldnames=['name', 'id']):
        relation_map[row['name']] = int(row['id'])

entity_emb = np.load('../data/embed/DRKG_TransE_l2_entity.npy')
rel_emb = np.load('../data/embed/DRKG_TransE_l2_relation.npy')

entity_emb.shape, rel_emb.shape

((97238, 400), (107, 400))

## Map drugs and diseases to embedding indices

In [5]:
drug_ids = [entity_map[d] for d in drug_list if d in entity_map]
disease_ids = [entity_map[d] for d in brain_injury_disease_list if d in entity_map]
treatment_rid = [relation_map[r] for r in treatment_relations if r in relation_map]

## Score all drugs against brain injury disease nodes

For each treatment relation type and each disease node, compute TransE scores across all candidate drugs, then aggregate.

In [6]:
gamma = 12.0

def transE_l2(head, rel, tail):
    return gamma - torch.norm(head + rel - tail, p=2, dim=-1)

drug_ids_tensor = torch.tensor(drug_ids).long()
drug_emb = torch.tensor(entity_emb[drug_ids_tensor])
treatment_embs = [torch.tensor(rel_emb[rid]) for rid in treatment_rid]

scores_per_disease = []
dids = []

for treatment_emb in treatment_embs:
    for disease_id in disease_ids:
        disease_emb = torch.tensor(entity_emb[disease_id])
        score = fn.logsigmoid(transE_l2(drug_emb, treatment_emb, disease_emb))
        scores_per_disease.append(score)
        dids.append(drug_ids_tensor)

scores = torch.cat(scores_per_disease)
dids = torch.cat(dids)

## Rank and get top 100

In [7]:
idx = torch.flip(torch.argsort(scores), dims=[0])
scores = scores[idx].numpy()
dids = dids[idx].numpy()

_, unique_indices = np.unique(dids, return_index=True)
topk_indices = np.sort(unique_indices)[:100]

results_df = pd.DataFrame({
    'rank': range(1, 101),
    'drug': [entity_id_map[int(d)] for d in dids[topk_indices]],
    'score': scores[topk_indices]
})

results_df.head(20)

,rank,drug,score
0,1,Compound::DB01109,-0.075181
1,2,Compound::DB01240,-0.082126
2,3,Compound::DB00682,-0.089073
3,4,Compound::DB09341,-0.102553
4,5,Compound::DB00974,-0.104952
5,6,Compound::DB00624,-0.114745
6,7,Compound::DB12290,-0.114957
7,8,Compound::DB01065,-0.120767
8,9,Compound::DB00328,-0.122030
9,10,Compound::DB00641,-0.122662


## Save results

In [8]:
import os
os.makedirs('../results', exist_ok=True)
results_df.to_csv('../results/transE_baseline_top100.csv', index=False)

In [14]:
import pubchempy as pcp
from tqdm import tqdm

manual_fixes = {
    'Compound::DB01109': 'Heparin',
    'Compound::DB09341': 'Glucose',
    'Compound::DB00770': 'Alprostadil',
}

def get_drug_name(drugbank_id):
    try:
        results = pcp.get_compounds(drugbank_id, 'name')
        if results:
            syns = results[0].synonyms
            return syns[0] if syns else 'unknown'
        return 'unknown'
    except:
        return 'unknown'

top20['drug_name'] = top20.apply(
    lambda row: manual_fixes.get(row['drug'], row['drug_name']), axis=1
)

top20[['rank', 'drug_name', 'drug', 'score']]

top20.to_csv('../results/transE_baseline_top20_named.csv', index=False)